# 🪖⚒️Construction Site Safety: A Beginner's EDA 🔎

**This is my first public notebook, I have tried to include everything I have learned so far. Please do let me know about any improvements I need to make at my end :D**

There have been many incidents of accidents on construction site, where workers succumbed to various intensities of injuries and even fatalities. Some common injuries include: 
- Burns 
- Scrapes
- Amputations
- Hearing loss 
- Head injuries 

and so on. Most of these injuries are caused due to improper guidance and lack of appropriate Personal Protective Equipment (PPE) kits.

Construction site managers and even some workers dont take these setbacks and incidents seriously thus making the construction site dangerous for others. In this notebook, we will try to assess a dataset on Construction Site Safety Image Dataset uploaded by Roboflow. The dataset can be accessed here: [**Construction Site Safety Image Dataset**](https://universe.roboflow.com/roboflow-universe-projects/construction-site-safety)

We will first do an exploratory data analysis on the CSS Dataset and try to see the distribution of `train`, `valid` and `test` sets. We will also visualize the samples, and run a custom object detection model to detect one of the following PPE classes: 

**['Hardhat', 'Mask', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest', 'Person', 'Safety Cone', 'Safety Vest', 'machinery', 'vehicle']** using YoloV8.

**`Note`** This is an introductory notebook, I have tried to include my learnings in a beginner-friendly manner.

# Import Libraries

We will import the following libraries for EDA and visualizations. 
- `os` and `glob` for accessing file directories 
- `numpy` for array operations
- `pandas` for dataframes, metadata creation
- `PIL` and `cv2` for Image oparations and visualization
- `matplotlib` for plotting results and class distributions

In [ ]:
%matplotlib inline
import os, glob
import tqdm
import numpy as np
import pandas as pd
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

### 🔑🔑 Key Tip 

Always start by mentioning the input and output path names, for future reference, access and usage. Input path names include all those paths, which are required for providing input to your ML model. Output path is where you save all your intermediate or final results.
Some outputs that you might save after training a model are :
- `csv` file which mentions the results, or predictions
- `model` file in `.h5`, `.ckpt`, '.pt` format
- `image` file in `.jpg` or `.png` format

Kaggle allows you to save around 19.5 GB of output files, after running your notebook. It also lets you save a working instance of the notebook via `Quick Save` which stores all the output directory contents.

**Yet another tip:** Try to maintain **uniformity** in naming variables. For instance, if you have mentioned the validation set path as `valid_path` make sure you use `valid` more than shortcuts `val` throughout as it reduces scope of errors.

In [ ]:
# Data path
data_path = 'C:/Users/HP/OneDrive/Desktop/PPE_Project/archive/css-data'
# Train, Valid and Test path
train_path = os.path.join(data_path, 'train')
valid_path = os.path.join(data_path, 'valid')
test_path = os.path.join(data_path, 'test')
# For saving results
output_path = 'C:/Users/HP/OneDrive/Desktop/PPE_Project/archive/results_yolov8n_100e/kaggle/working'
# We can access both images and labels
folders = ['images', 'labels']
print("Data Path: {}\nTrain Path: {}\nValid Path: {}\nTest Path: {}\nOutput Path: {}".format(data_path, train_path, valid_path, test_path, output_path))

# 📅 ℹ️ Metadata
Metadata literally means data about data. We will create metadata (`csv` file) which contains all the information about the file directory and the split information (which file is train, valid or test). We will not include raw image information as it will make the csv file very heavy and it is unncessary.

Later only through this metadata we can make sure whether a file is in the train, valid or test set and what are the annotations for the file. It is also very helpful in visualizing results. In scenarios, where dataset is not provided, heavy metadata files are provided (multiple CSV files) which is easier if you are dealing with say 1 million images. 


In [ ]:
# Initialize dictionaries of training and classes
train_dict = dict(train=0, valid=1, test=2)
path_dict = [train_path, valid_path, test_path]
class_names = ['Hardhat', 'Mask', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest', 'Person', 'Safety Cone', 'Safety Vest', 'machinery', 'vehicle']
class_dict = dict(zip(range(len(class_names)), class_names))
print(class_dict)

In [ ]:
## Get filenames and labels information
# Sorting the filenames will make the labels and images in same order 
train_filenames = sorted(os.listdir(os.path.join(train_path, folders[0])))
valid_filenames = sorted(os.listdir(os.path.join(valid_path, folders[0])))
test_filenames = sorted(os.listdir(os.path.join(test_path, folders[0])))
train_labels = sorted(os.listdir(os.path.join(train_path, folders[1])))
valid_labels = sorted(os.listdir(os.path.join(valid_path, folders[1])))
test_labels = sorted(os.listdir(os.path.join(test_path, folders[1])))

In [ ]:
## One liner for the above code
# We can also use list comprehension for this
t_f, v_f, te_f = [sorted(os.listdir(os.path.join(path_dict[i], folders[0]))) for i in range(len(path_dict))]
t_l, v_l, te_l = [sorted(os.listdir(os.path.join(path_dict[i], folders[1]))) for i in range(len(path_dict))]

In [ ]:
# Check whether both gives same results in filenames
train_filenames==t_f, valid_filenames==v_f, test_filenames==te_f

In [ ]:
# Check whether both gives same results in labels
train_labels==t_l, valid_labels==v_l, test_labels==te_l

## ⬜ Check these things
1. Length of **training/validation/test filenames** are same as **training/validation/test labels**

2. Order of **training/validation/test filenames** are same as **training/validation/test labels**

3. Leakage of files and duplicate entries in **training/validation/test filenames**

### ✅ Check lengths

In [ ]:
# Return lengths of all filenames
print("Total Train Files: {}\nTotal Valid Files: {}\nTotal Test Files:{}".format(len(train_filenames), len(valid_filenames), len(test_filenames)))

In [ ]:
# Check whether filenames and labels are of same length
len(train_filenames)==len(train_labels), len(valid_filenames)==len(valid_labels), len(test_filenames)==len(test_labels)

### ✅ Check Order
We will check the order of filenames and labels to make sure that each image has the same label annotation name as that of the image. This will ensure that we are not using annotations and label information for a different image.

In [ ]:
# Check order in filenames and labels in all splits
[item.split('.')[0] for item in train_filenames]==[item.split('.')[0] for item in train_labels],\
[item.split('.')[0] for item in valid_filenames]==[item.split('.')[0] for item in valid_labels],\
[item.split('.')[0] for item in test_filenames]==[item.split('.')[0] for item in test_labels]

### ✅ Check Leakage
We will check whether any filename is present in more than one set. This is known as leakage in datasets. If some filenames are common in more than one set say training or validation, then the training will not be dependable. 

Manytimes, large datasets from different competitions have manual annotators. We should always check these three things before proceeding with EDA or pre-processing.

In [ ]:
set(train_filenames).intersection(set(valid_filenames)),\
set(valid_filenames).intersection(set(test_filenames)),\
set(test_filenames).intersection(set(train_filenames))

In [ ]:
df = pd.DataFrame()
df['filenames'] = train_filenames + valid_filenames + test_filenames
df['labelnames'] = train_labels + valid_labels + test_labels
df['train_id'] = [0]*len(train_filenames) + [1]*len(valid_filenames) + [2]*len(test_filenames)

In [ ]:
df.head()

### ✅ Check Duplicate Entries

In [ ]:
# No duplicate entries found
df.filenames.duplicated().value_counts()

In [ ]:
# Count of train valid and test sets
df.train_id.value_counts()

In [ ]:
df.train_id.value_counts().plot(kind = 'bar', title = 'Train-Val-Test Split')

## Visualize Class Distribution
In this dataset, each image consists of more than 1 class annotation in general. So we need to check the total number of annotations in each image only then we can visualize an overall class distribution.

There are three cases when we read the annotation file. 

**1. Annotation file is empty.**

`len(annotation)===0`
We will skip these files during training and evaluation of our object detection model.

**2. Annotation file is having only one annotation.** 

`len(annotation.shape)==1`
We will reshape the array and get the annotations.

**3. Annotation file is having multiple annotations.** 

`len(annotation)>1`
We will get the annotations direclty.

In [ ]:
# Split list
train_keys = list(train_dict.keys())

In [ ]:
# Complete path for annotation_files
annotation_files = (data_path + '/' + df.train_id.map(lambda x: train_keys[x]) + '/' + folders[1]
                    + '/' + df.labelnames).tolist()
t_id = df.train_id.tolist()
counts = []
invalid_idx = []
is_annotated = []
for idx, annotation_file in tqdm.tqdm(enumerate(annotation_files)):
    annotation = np.loadtxt(annotation_file)
    if len(annotation)==0:
        invalid_idx.append(idx)
        is_annotated.append(-1)
        counts.append([])
        continue
    if len(annotation.shape)==1:
        annotation = annotation.reshape(1, -1)
    counts.append(annotation[:,0].astype(int))
    is_annotated.append(1)
df['is_annotated'] = is_annotated

In [ ]:
df.is_annotated.value_counts()

In [ ]:
# Create a count_dict which holds class counts per split (train/valid/test)
count_list = [np.unique(item, return_counts = True) for item in counts]
count_keys = [item[0] for item in count_list]
count_values = [item[1] for item in count_list]
count_dict = []
for ck,cv in zip(count_keys, count_values):
    count_dict.append(dict([(key,value) for key, value in zip(ck,cv)]))
df['count_dict'] = count_dict

In [ ]:
from collections import Counter
train_count = df[df.train_id==0].count_dict.apply(lambda x: Counter(x)).sum()
valid_count = df[df.train_id==1].count_dict.apply(lambda x: Counter(x)).sum()
test_count = df[df.train_id==2].count_dict.apply(lambda x: Counter(x)).sum()

In [ ]:
# We will normalize the counts to get a correct visualization, since there is huge difference
# between the counts of train valid and test splits
train_count = {key:value/sum(train_count.values()) for key, value in train_count.items()}
valid_count = {key:value/sum(valid_count.values()) for key, value in valid_count.items()}
test_count = {key:value/sum(test_count.values()) for key, value in test_count.items()}

In [ ]:
print("Train Class Distribution Dict: {}\n\nValid Class Distribution Dict: {}\n\nTest Class Distribution Dict: {}"
     .format(train_count, valid_count, test_count))

In [ ]:
df_count = pd.DataFrame()
df_count = pd.DataFrame({'train':train_count, 'valid': valid_count, 'test': test_count}).sort_index()

In [ ]:
df_count['idx'] = pd.Series(df_count.index)

In [ ]:
df_count.plot(y = ['train', 'valid', 'test'], kind = 'bar', title = 'Train Valid Split Distribution')
plt.legend(['Train', 'Valid', 'Test'])

We can see the normalized counts of all the classes are more or less balanced, this will ensure that we are not seeing some classes more oftenly than others during training as the distribution is almost same in all classes (among all splits).

# Visualize Samples
Let us visualize some samples from the dataset and their annotations. Since we have all the annotations in YoloV8 format we will start with converting these annotation files to bounding box coordinates.

YoloV8 format for annotation has 5 entries in each line of the txt file. 
- Class Label (c)
- Bounding Box's center (X coordinate)
- Bounding Box's center (Y coordinate)
- Width of Bounding Box (w)
- Height of Bounding Box (h)

The bounding box can be calculated from the last 4 entries in the annotation file.

In [ ]:
def yolo_annotation_to_bbox(annotation, img_height, img_width):
    """
    Converts Yolo annotations to bounding box coordinates
    Input:
    annotation: str, annotation file in .txt format
    img_height: int, image height
    img_width: int, image width
    Output:
    class: list, List of labels in the image
    bbox_list: list, List of bounding boxes in an image
    """
    sh = annotation.shape
    if len(sh)==0:
        print("No bounding box found")
    if len(sh)==1:
        annotation = annotation.reshape(1, -1)
    num_bbox = len(annotation)
    bbox_list = []
    for idx in range(num_bbox):
        c_x, c_y, w, h = annotation[idx][1:]
        x1 = ((c_x - w/2)*img_width).astype(int)
        x2 = ((c_x + w/2)*img_width).astype(int)
        y1 = ((c_y - h/2)*img_height).astype(int)
        y2 = ((c_y + h/2)*img_height).astype(int)
        bbox_list.append([x1, y1, x2, y2])
    return bbox_list

In [ ]:
### invalid_files = (data_path + '/' + df.train_id[invalid_idx].apply(lambda x: train_keys[x]) + '/' + folders[0] + '/' + df.filenames[invalid_idx])
invalid_files = df.filenames[invalid_idx]
def visualize_samples(mode = 'train', n_samples = 12):
    """
    Plots 'n_samples' plots from train/valid/test split
    Input:
    mode: 'str' can take values from 'train'/'valid','test'
    n_samples: 'int'
    """
    # We will visualize only those files which have annotations 
    indices = df[(~df.filenames.isin(invalid_files))&(df.train_id==1)].sample(n_samples).index
    filenames = (data_path + '/' + df.train_id[indices].apply(lambda x: train_keys[x]) + '/' + folders[0] + '/' + df.filenames[indices]).tolist()
    annotations = (data_path + '/' + df.train_id[indices].apply(lambda x: train_keys[x]) + '/' + folders[1] + '/' + df.labelnames[indices]).tolist()
    plt.figure(figsize = (21, 11))
    plt.title('{} Set Samples'.format(mode.upper()))
    for idx in range(len(filenames)):
        image = np.array(Image.open(filenames[idx]))
        height, width, _ = image.shape 
        annotation = np.loadtxt(annotations[idx])
        bbox_list = yolo_annotation_to_bbox(annotation, height, width)
        if len(annotation.shape)==1:
            annotation = annotation.reshape(1, -1)
        labels = [class_dict[item] for item in annotation[:,0].astype(int)]
        plt.subplot(3, 4, idx + 1)
        for label, bbox in zip(labels, bbox_list):
            x1, y1, x2, y2 = bbox
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(image, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
        plt.imshow(image)
    plt.tight_layout()
    plt.show()

In [ ]:
visualize_samples(mode = 'train', n_samples = 12)

In [ ]:
visualize_samples(mode = 'valid', n_samples = 12)

In [ ]:
visualize_samples(mode = 'test', n_samples = 12)

# 🧑‍💻 👓Run Custom Object Detection Model
After the visualization of dataset it is time to train our custom object detection. We will use YoloV8 to train our custom object detection model. More information on YoloV8 here: [**Ultralytics Docs**](https://docs.ultralytics.com)

## Install Ultralytics

In [ ]:
print('Skipping ultralytics install; package already present in local venv')
from ultralytics import YOLO


## Create YAML file

We will create a YAML file which will be used by the YoloV8 CLI for information. It mentions the `train`, `valid`, `test` directories and the `nc` (number of classes) for training the model

In [ ]:
import yaml
ppe_data = dict(train = train_path,
                    val = valid_path,
                    test = test_path,
                   nc = len(class_names),
                   names = class_names)
with open('ppe_data.yaml', 'w') as output:
    yaml.dump(ppe_data, output, default_flow_style = True)

In [ ]:
from pathlib import Path
print(Path(r'C:/Users/HP/OneDrive/Desktop/PPE_Project/archive/results_yolov8n_100e/kaggle/working/ppe_data.yaml').read_text())


## Train a custom model

We will use the Command Line Interface (CLI) command for training a custom model using YoloV8. The arguments provided depends on our tasks and training. We will train the model for 250 epochs on a pre-trained model (on COCO128 data) for our PPE Detection dataset.

In [ ]:
!yolo task=detect mode=train epochs=300 data='C:/Users/HP/OneDrive/Desktop/PPE_Project/archive/results_yolov8n_100e/kaggle/working/ppe_data.yaml' model=yolov8m.pt imgsz=640 patience=50

In [ ]:
!zip -r results_yolov8n_100e.zip C:/Users/HP/OneDrive/Desktop/PPE_Project/archive/results_yolov8n_100e/kaggle/working

# 📈📊Visualize Results

In [ ]:
train_results_path = 'C:/Users/HP/OneDrive/Desktop/PPE_Project/archive/results_yolov8n_100e/kaggle/working/runs/detect/train/'
csv_results = train_results_path + 'results.csv'
image_results = train_results_path + '*.*'
df_results = pd.read_csv(csv_results)
df_results.head()

In [ ]:
plt.figure(figsize = (21, 11))
for result in glob.glob(image_results):
    ext = result.split('/')[-1].split('.')[-1]
    if (ext=='jpg')or(ext=='jpeg')or(ext=='png'):
        image = Image.open(result)
        image = np.array(image)
        plt.imshow(image)
        plt.title('{}'.format(result.split('/')[-1].split('.')[0]))
    plt.show()

# 🔮Prediction

In [ ]:
from ultralytics import YOLO
best_model = train_results_path + 'weights/best.pt'
test_results_path = 'C:/Users/HP/OneDrive/Desktop/PPE_Project/archive/css-data/test/images'
test_result_filenames = glob.glob(test_results_path + '/*.*')
model = YOLO(best_model)
results = model.predict(source= test_results_path, save = True, show=False, save_txt = True) 
for filename in test_result_filenames:
    results = model.predict(source = filename, save = True, show = False, save_txt = True)

## Final Accuracy Metrics

This cell shows the main YOLO validation metrics from the training run.


In [ ]:
from pathlib import Path
import pandas as pd

results_path = Path(r'C:\Users\HP\OneDrive\Desktop\PPE_Project\archive\results_yolov8n_100e\kaggle\working\runs\detect\train\results.csv')
df = pd.read_csv(results_path)
df.columns = df.columns.str.strip()
best_row = df.loc[df['metrics/mAP50-95(B)'].idxmax()]

print(f"Best epoch: {int(best_row['epoch'])}")
print(f"Precision: {best_row['metrics/precision(B)'] * 100:.2f}%")
print(f"Recall: {best_row['metrics/recall(B)'] * 100:.2f}%")
print(f"mAP@50: {best_row['metrics/mAP50(B)'] * 100:.2f}%")
print(f"mAP@50:95: {best_row['metrics/mAP50-95(B)'] * 100:.2f}%")
